# Set 7

## Dataset: FinancialPhraseBank

Source: Provided via course link (downloaded as CSV from Google Drive).

### Group Details:
Group No: 78

#### Group Members:
1.   SANJANA V KULKARNI - 2024DA04217
2.   BANDARU HAREESHA - 2024DA04089
3.   BHAVYA ARORA - 2023DA04058
4.   AVINASH GUPTA - 2023DC04310

---

### Objective


1.   Perform exploratory analysis and preprocessing on financial news headlines.
2.   Apply LDA to extract latent topics from the text corpus.
3.  Evaluate and visualize topics and syntactic dependencies for interpretability.

###How to run — Quick steps:

1.   Run the notebook top to bottom in Google Colab or Jupyter.
2.   Upload **all-data.csv** when prompted.
3.   The notebook performs EDA, preprocess text data, topic modeling, coherence evaluation, and visualization.
4.   Outputs are saved in the working directory.

###Notes/Assumptions:

1.   Only the first 2000 rows are used as per instructions.
2.   Topic modeling is performed on the **News Headline** column only.
3.   Number of topics is fixed at 10.
4.   Dependency parsing is shown for sentences with at least 10 words.


## Install and Import Libraries

In [8]:
!pip install pandas numpy matplotlib seaborn scikit-learn nltk gensim spacy pyLDAvis
!python -m spacy download en_core_web_sm


zsh:1: command not found: pip
zsh:1: command not found: python


In [ ]:
#Import and Setup

# Install missing packages
%pip install -q gensim spacy

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import nltk
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

from gensim.corpora.dictionary import Dictionary
from gensim.models.coherencemodel import CoherenceModel

import spacy
from spacy import displacy

from google.colab import files
import io

print("✓ All packages imported successfully!")

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
  distutils: /private/var/folders/md/kf36c0kn65d0c_b5ljtdmwb00000gn/T/pip-build-env-mcexunqk/normal/lib/python3.9/site-packages
  sysconfig: /Library/Python/3.9/site-packages
  distutils: /private/var/folders/md/kf36c0kn65d0c_b5ljtdmwb00000gn/T/pip-build-env-mcexunqk/normal/lib/python3.9/site-packages
  sysconfig: /Library/Python/3.9/site-packages
  user = False
  home = None
  root = None
  prefix = '/private/var/folders/md/kf36c0kn65d0c_b5ljtdmwb00000gn/T/pip-build-env-mcexunqk/normal'
  distutils: /private/var/folders/md/kf36c0kn65d0c_b5ljtdmwb00000gn/T/pip-build-env-mcexunqk/overlay/lib/python3.9/site-packages
  sysconfig: /Library/Python/3.9/site-packages
  distutils: /private/var/folders/md/kf36c0kn65d0c_b5ljtdmwb00000gn/T/pip-build-env-mcexunqk/overlay/lib/python3.9/site-packages
  sysconfig: /Library/Python/3.9/site-packages
  user = False
  home = N

KeyboardInterrupt: 

## Load Dataset

In [ ]:
#Upload File , Load Dataset and Use first 2000 rows

uploaded = files.upload()
df = pd.read_csv('all-data.csv', encoding='latin-1')
df = df.head(2000)
df.head()




In [ ]:
df.columns = ['Sentiment', 'News Headline']
df.head()

## Task -1 EDA & Preprocessing

### Sentiment Distribution

In [ ]:
sentiment_counts = df['Sentiment'].value_counts()
sentiment_counts



In [ ]:
plt.figure(figsize=(6,4))
sns.barplot(x=sentiment_counts.index, y=sentiment_counts.values)
plt.title("Sentiment Distribution")
plt.ylabel("Count")
plt.savefig("sentiment_distribution.png", dpi=300)
plt.show()


### Text Statistics

In [ ]:
df['char_length'] = df['News Headline'].apply(len)
df['word_count'] = df['News Headline'].apply(lambda x: len(x.split()))

df[['char_length','word_count']].describe()


In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['word_count'], bins=30)
plt.title("Word Count Distribution")
plt.savefig("text_length_distribution.png", dpi=300)
plt.show()


## Text Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words and len(w) > 2]
    return tokens

df['tokens'] = df['News Headline'].apply(preprocess)
df['clean_text'] = df['tokens'].apply(lambda x: ' '.join(x))


### Most Common Words

In [ ]:
all_words = [word for tokens in df['tokens'] for word in tokens]
word_freq = Counter(all_words)

top_words = word_freq.most_common(10)
top_words


In [ ]:
words, counts = zip(*top_words)
plt.figure(figsize=(8,4))
sns.barplot(x=list(counts), y=list(words))
plt.title("Top 10 Most Common Words")
plt.savefig("top_words.png", dpi=300)
plt.show()


## TASK 2 – LDA TOPIC MODELING (10 TOPICS)

In [ ]:
vectorizer = CountVectorizer(max_df=0.9, min_df=10)
dtm = vectorizer.fit_transform(df['clean_text'])

lda = LatentDirichletAllocation(
    n_components=10,
    random_state=42,
    learning_method='online',
    max_iter=50
)

lda.fit(dtm)


In [ ]:
feature_names = vectorizer.get_feature_names_out()

def print_topics(model, feature_names, n_top_words=5):
    topics = []
    for idx, topic in enumerate(model.components_):
        words = [feature_names[i] for i in topic.argsort()[:-n_top_words-1:-1]]
        topics.append(words)
        print(f"Topic {idx+1}: {', '.join(words)}")
    return topics

topics = print_topics(lda, feature_names)


## TASK 3 – COHERENCE SCORE (UMASS)

In [ ]:
dictionary = Dictionary(df['tokens'])
corpus = [dictionary.doc2bow(text) for text in df['tokens']]

coherence_model = CoherenceModel(
    topics=topics,
    corpus=corpus,
    dictionary=dictionary,
    coherence='u_mass'
)

coherence_score = coherence_model.get_coherence()
coherence_score


## TASK 4 – TOPIC VISUALIZATION

### Topic Distribution

In [ ]:
topic_probs = lda.transform(dtm)
dominant_topics = topic_probs.argmax(axis=1)

topic_dist = pd.Series(dominant_topics).value_counts().sort_index()

plt.figure(figsize=(8,4))
sns.barplot(x=topic_dist.index+1, y=topic_dist.values)
plt.title("Topic Distribution Across Documents")
plt.xlabel("Topic")
plt.ylabel("Documents")
plt.savefig("topic_distribution.png", dpi=300)
plt.show()


## TASK 5 – DEPENDENCY PARSER (2 SENTENCES ≥ 10 WORDS)

In [ ]:
nlp = spacy.load("en_core_web_sm")

sentences = df[df['word_count'] >= 10]['News Headline'].sample(2, random_state=42).tolist()
sentences


### Visualize Dependency Trees

In [ ]:
for i, sent in enumerate(sentences, 1):
    doc = nlp(sent)
    svg = displacy.render(doc, style='dep', jupyter=False)
    with open(f"dependency_parse_sentence_{i}.html", "w", encoding="utf-8") as f:
        f.write(svg)
    display(displacy.render(doc, style='dep'))


## SAVE OUTPUT FILES

In [ ]:
df.to_csv("preprocessed_financial_data.csv", index=False)


### Summary
This assignment explored financial news headlines using NLP techniques.
After preprocessing, LDA was applied to extract 10 latent topics and
topic coherence was evaluated. Topic distributions and dependency
parses were visualized to improve interpretability of results.
